# Practice 1 — Interactive Personal Budget Manager
**Session 10 · Python I · Release 1**

AI-assisted practice notebook. It records the expected behavior, executable checks and captured results. All writes below use a temporary folder, not personal data. The implementation is in `src/budget_manager/`; the full automated test suite is in `tests/test_budget.py`.

## 1. Think → Predict

| Input / scenario | Predicted result |
|---|---|
| Add 100 EUR salary and 18.50 EUR food expense | 2 transactions; balance 81.50 EUR |
| Add -5 or 0 EUR | ValueError; existing records unchanged |
| Add an unknown expense category | ValueError; existing records unchanged |
| Save and load CSV | All records are identical after reload |
| Import invalid sample | Error identifies physical row 3 |
| Undo after restarting the CLI | Last record removed from disk; balance 100.00 EUR |

Design: CLI handles user interaction, services handle budget logic, validation checks the two entry boundaries and storage manages file I/O.

In [1]:
from decimal import Decimal
from pathlib import Path
from tempfile import TemporaryDirectory
import subprocess
import sys

from src.budget_manager.services import add_transaction, summarize
from src.budget_manager.storage import load_transactions, save_transactions

records = []
add_transaction(records, "2026-09-01", "income", "salary", 100)
add_transaction(records, "2026-09-04", "expense", "food", 18.50)
report = summarize(records)["EUR"]
assert len(records) == 2
assert report["income"] == Decimal("100.00")
assert report["expenses"] == Decimal("18.50")
assert report["balance"] == Decimal("81.50")
print(f"Normal case: balance = {report['balance']:.2f} EUR (PASS)")

Normal case: balance = 81.50 EUR (PASS)


## 2. Boundary cases

Negative and zero amounts and unknown categories must not change the list. Check the error message as well as the transaction count.

In [2]:
for amount, category, expected in [(-5, "food", "amount"),
                                    (0, "food", "amount"),
                                    (20, "unknown", "unknown expense category")]:
    before = records.copy()
    try:
        add_transaction(records, "2026-09-04", "expense", category, amount)
    except ValueError as error:
        assert expected in str(error)
        assert records == before
        print(f"Rejected ({amount}, {category}): {error} (PASS)")
    else:
        raise AssertionError("Invalid transaction was accepted")

Rejected (-5, food): amount must be > 0 and finite. (PASS)
Rejected (0, food): amount must be > 0 and finite. (PASS)
Rejected (20, unknown): unknown expense category 'unknown'; choose food, housing, transport, utilities, health, education, entertainment, other. (PASS)


## 3. Persistence and end-to-end evidence

Save the records, reload them and run the CLI in a separate process to show that undo persists beyond the original Python session.

In [3]:
with TemporaryDirectory() as folder:
    path = Path(folder) / "transactions.csv"
    save_transactions(path, records)
    assert load_transactions(path) == records
    print("CSV round trip: all records identical (PASS)")

    for command in ("summary", "undo", "summary"):
        result = subprocess.run([sys.executable, "budget.py", "--file", str(path), command],
                                capture_output=True, text=True, check=True)
        print(f"\n$ budget.py {command}\n{result.stdout.strip()}")
    assert len(load_transactions(path)) == 1
    assert summarize(load_transactions(path))["EUR"]["balance"] == Decimal("100.00")
    print("Undo persisted after restarting the CLI (PASS)")

CSV round trip: all records identical (PASS)



$ budget.py summary
Currency: EUR
Income:   100.00
Expenses: 18.50
Balance:  81.50
Expenses by category:
  food: 18.50



$ budget.py undo
Undid T002.



$ budget.py summary
Currency: EUR
Income:   100.00
Expenses: 0.00
Balance:  100.00
Undo persisted after restarting the CLI (PASS)


In [4]:
try:
    load_transactions(Path("data/sample_invalid_transactions.csv"))
except ValueError as error:
    assert "Row 3: amount must be > 0" in str(error)
    print(f"Invalid CSV: {error} (PASS)")
else:
    raise AssertionError("Invalid CSV was accepted")

Invalid CSV: Row 3: amount must be > 0 and finite. (PASS)


## 4. Exit ticket — suggested reflection to review and personalize

- **What was verified?** The normal case produces a balance of 81.50 EUR and, after a persistent undo, the reloaded balance is 100.00 EUR.
- **Which boundary influenced the design?** Imported CSV data can contain the same invalid amounts as CLI input, so both paths share the same validation function. A failed import does not silently discard invalid records.
- **What could be refactored next?** The storage layer could gain locking or a database transaction strategy for concurrent updates. Release 1 is intentionally a single-user CLI.

See `TEST_EVIDENCE.txt` for the broader automated test run and `AI_DISCLOSURE.md` for AI provenance.